# Complete subject-specific EMG analysis — BAMon

This notebook performs the complete, leakage-controlled analysis for the
**active-resistance BAMon STANDARD 3x** condition.

It:

1. audits the source files and trial structure;
2. constructs independent Earth and simulated-Moon datasets;
3. records the exact chronological training/validation/test partitions;
4. compares a training-mean predictor, Ridge, Random Forest, and XGBoost
   using identical held-out test windows;
5. saves software versions and selected hyperparameters;
6. reports participant-level and robust-range-normalised metrics;
7. flags low-variance and extreme results for data-quality review without
   automatically excluding them;
8. creates participant-distribution, model-comparison, and paper-trace figures.

Run every cell from top to bottom. The analysis remains subject-specific:
training and testing use different chronological sections from the same
participant and recording, not previously unseen participants.

## 1. Install the analysis dependencies

Versions are deliberately recorded later in the notebook. Do not update
packages between BAMon and BAMoff runs if the results will be compared.

In [ ]:
!pip -q install numpy pandas scipy scikit-learn matplotlib xgboost tensorflow

## 2. Mount Google Drive and load the shared pipeline

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import sys
import importlib
import inspect
from google.colab import files

PIPELINE_FILENAME = 'emg_exosuit_pipeline_colab.py'
PIPELINE_PATH = os.path.join('/content', PIPELINE_FILENAME)

if not os.path.isfile(PIPELINE_PATH):
    print(f'Upload {PIPELINE_FILENAME}')
    uploaded = files.upload()
    assert PIPELINE_FILENAME in uploaded, (
        f'Please upload {PIPELINE_FILENAME}'
    )

sys.path.insert(0, '/content')
if 'emg_exosuit_pipeline_colab' in sys.modules:
    del sys.modules['emg_exosuit_pipeline_colab']

import emg_exosuit_pipeline_colab as pipeline
importlib.reload(pipeline)
from emg_exosuit_pipeline_colab import *

split_signature = inspect.signature(choose_subject_specific_masks)
assert 'purge_windows' in split_signature.parameters, (
    'The uploaded pipeline is too old: choose_subject_specific_masks '
    'must support purge_windows. Upload the corrected pipeline.'
)

print('Pipeline loaded from:', pipeline.__file__)

## 3. Configuration

Change `EMG_FOLDER` or `SUIT_FOLDER` only if the Drive folders have
different names. The output directory is new, so earlier results are not
overwritten.

In [ ]:
import json
import platform
import hashlib
import shutil
import numpy as np
import pandas as pd
import scipy
import sklearn
import xgboost
import tensorflow
import matplotlib
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

SUIT_STATE = 'BAMon'
CONDITION = 'BAMon STANDARD 3x'

EMG_FOLDER = '/content/drive/MyDrive/data/emg_csv'
SUIT_FOLDER = '/content/drive/MyDrive/data/suit2'
RESULTS_ROOT = '/content/drive/MyDrive/results_BAMon_complete_analysis'

cfg = ExperimentConfig(
    emg_folder=EMG_FOLDER,
    suit_folder=SUIT_FOLDER,
    results_root=RESULTS_ROOT,
    conditions=[CONDITION],
    target_muscles=ALL_MUSCLES.copy(),
    suit_features=[
        'forcefront',
        'forceback',
        'pressurefront',
        'pressureback',
    ],
    target_fs=100.0,
    window_seconds=0.1,
    window_overlap=0.5,
    target_aggregation='mean',
    seed=42,
    run_handcrafted_models=True,
    run_cnn_lstm=False,
)

assert os.path.isdir(EMG_FOLDER), f'EMG folder not found: {EMG_FOLDER}'
assert os.path.isdir(SUIT_FOLDER), f'SUIT folder not found: {SUIT_FOLDER}'
os.makedirs(RESULTS_ROOT, exist_ok=True)

archived_pipeline_path = os.path.join(
    RESULTS_ROOT,
    'emg_exosuit_pipeline_colab.py',
)
shutil.copy2(pipeline.__file__, archived_pipeline_path)
with open(archived_pipeline_path, 'rb') as handle:
    pipeline_sha256 = hashlib.sha256(handle.read()).hexdigest()

print('Suit state:', SUIT_STATE)
print('Source condition:', CONDITION)
print('Results:', RESULTS_ROOT)

## 4. Save the software and analysis configuration

In [ ]:
environment = {
    'python': platform.python_version(),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'scipy': scipy.__version__,
    'scikit_learn': sklearn.__version__,
    'xgboost': xgboost.__version__,
    'tensorflow': tensorflow.__version__,
    'matplotlib': matplotlib.__version__,
    'pipeline_file': pipeline.__file__,
    'archived_pipeline_sha256': pipeline_sha256,
}

analysis_configuration = {
    'suit_state': SUIT_STATE,
    'source_condition': CONDITION,
    'target_muscles': cfg.target_muscles,
    'suit_features': cfg.suit_features,
    'target_sampling_frequency_hz': cfg.target_fs,
    'window_seconds': cfg.window_seconds,
    'window_overlap': cfg.window_overlap,
    'window_stride_seconds': (
        cfg.window_seconds * (1.0 - cfg.window_overlap)
    ),
    'target_aggregation': cfg.target_aggregation,
    'test_fraction_before_purge': 0.20,
    'validation_fraction_of_remaining_before_purge': 0.20,
    'purge_windows_per_partition_edge': 20,
    'random_seed': cfg.seed,
    'ridge_grid': cfg.ridge_alphas,
    'random_forest_grid': cfg.rf_grid,
    'xgboost_grid': cfg.xgb_grid,
    'xgboost_objective': 'reg:squarederror',
    'xgboost_reg_lambda': 1.0,
    'hyperparameter_selection_metric': 'validation MAE',
    'early_stopping': False,
}

with open(
    os.path.join(RESULTS_ROOT, 'software_versions.json'),
    'w',
    encoding='utf-8',
) as handle:
    json.dump(environment, handle, indent=2)

with open(
    os.path.join(RESULTS_ROOT, 'analysis_configuration.json'),
    'w',
    encoding='utf-8',
) as handle:
    json.dump(analysis_configuration, handle, indent=2)

display(pd.Series(environment, name='version'))

## 5. Load and audit the condition-specific files

These assertions prevent accidental mixing of BAMon and BAMoff files and
confirm that Earth and Moon trials exist in both sources.

In [ ]:
emg_df, suit_df = load_condition_data(cfg)

source_summary = pd.DataFrame({
    'dataset': ['EMG', 'SUIT'],
    'rows': [len(emg_df), len(suit_df)],
    'subjects': [
        emg_df['Subject'].nunique(),
        suit_df['Subject'].nunique(),
    ],
    'trials': [
        emg_df['TrialID'].nunique(),
        suit_df['TrialID'].nunique(),
    ],
    'earth_trials': [
        emg_df.loc[
            emg_df['Gravity'].astype(str).str.lower().eq('earth'),
            'TrialID',
        ].nunique(),
        suit_df.loc[
            suit_df['Gravity'].astype(str).str.lower().eq('earth'),
            'TrialID',
        ].nunique(),
    ],
    'moon_trials': [
        emg_df.loc[
            emg_df['Gravity'].astype(str).str.lower().eq('moon'),
            'TrialID',
        ].nunique(),
        suit_df.loc[
            suit_df['Gravity'].astype(str).str.lower().eq('moon'),
            'TrialID',
        ].nunique(),
    ],
})

assert source_summary['earth_trials'].min() > 0, 'Earth trials are missing.'
assert source_summary['moon_trials'].min() > 0, 'Moon trials are missing.'

emg_trials = set(emg_df['TrialID'].unique())
suit_trials = set(suit_df['TrialID'].unique())
assert emg_trials == suit_trials, (
    'EMG and SUIT trial identifiers do not match exactly.'
)

trial_audit = (
    emg_df[['Subject', 'Gravity', 'TrialID']]
    .drop_duplicates()
    .sort_values(['Gravity', 'Subject', 'TrialID'])
    .reset_index(drop=True)
)
trial_audit.insert(0, 'suit_state', SUIT_STATE)

source_summary.to_csv(
    os.path.join(RESULTS_ROOT, 'source_summary.csv'),
    index=False,
)
trial_audit.to_csv(
    os.path.join(RESULTS_ROOT, 'trial_audit.csv'),
    index=False,
)

display(source_summary)
display(trial_audit)

## 6. Build independent Earth and Moon window datasets

In [ ]:
def condition_frames(gravity):
    emg_part = emg_df[
        emg_df['Gravity'].astype(str).str.lower().eq(gravity.lower())
    ].copy()
    suit_part = suit_df[
        suit_df['Gravity'].astype(str).str.lower().eq(gravity.lower())
    ].copy()
    assert not emg_part.empty and not suit_part.empty
    return emg_part, suit_part


def build_condition_dataset(gravity, muscle):
    emg_part, suit_part = condition_frames(gravity)
    data = build_window_dataset(
        emg_df=emg_part,
        suit_df=suit_part,
        muscle=muscle,
        suit_features=cfg.suit_features,
        fs=cfg.target_fs,
        window_seconds=cfg.window_seconds,
        overlap=cfg.window_overlap,
        target_aggregation=cfg.target_aggregation,
        min_trial_samples=cfg.min_trial_samples,
    )

    expected = f'_{gravity.lower()}_'
    assert all(expected in str(t).lower() for t in data.trial_ids)
    assert np.isfinite(data.X_features).all(), (
        f'Non-finite predictors: {muscle}, {gravity}'
    )
    assert np.isfinite(data.y).all(), (
        f'Non-finite targets: {muscle}, {gravity}'
    )
    return data


datasets = {}
dataset_rows = []

for muscle in cfg.target_muscles:
    for gravity in ['Earth', 'Moon']:
        data = build_condition_dataset(gravity, muscle)
        datasets[(muscle, gravity)] = data

        dataset_rows.append({
            'suit_state': SUIT_STATE,
            'muscle': muscle,
            'gravity': gravity,
            'windows': len(data.y),
            'subjects': len(np.unique(data.subjects)),
            'trials': len(np.unique(data.trial_ids)),
            'window_samples': data.window_samples,
            'features': data.X_features.shape[1],
            'target_mean': float(np.mean(data.y)),
            'target_std': float(np.std(data.y)),
        })

dataset_summary = pd.DataFrame(dataset_rows)
dataset_summary.to_csv(
    os.path.join(RESULTS_ROOT, 'dataset_summary.csv'),
    index=False,
)

feature_names = pd.DataFrame({
    'feature_name': next(iter(datasets.values())).feature_names
})
feature_names.to_csv(
    os.path.join(RESULTS_ROOT, 'feature_names_124.csv'),
    index=False,
)

display(dataset_summary)
print('Number of extracted features:', len(feature_names))

## 7. Record the exact chronological partitions

The nominal split before purging is approximately 64% training, 16%
validation, and 20% testing. Twenty window positions are then removed
from each adjacent edge of every partition boundary. With a 0.05-s
stride, the retained window timestamps are separated by approximately
two seconds. The CSV produced below is the authoritative record of the
actual counts and gaps.

In [ ]:
split_rows = []

for (muscle, gravity), data in datasets.items():
    for subject in np.unique(data.subjects):
        subject_mask = data.subjects == subject
        trial_ids = data.trial_ids[subject_mask]
        times = data.window_times[subject_mask]

        train_mask, val_mask, test_mask = (
            choose_subject_specific_masks(
                trial_ids,
                test_fraction=0.20,
                val_fraction=0.20,
                purge_windows=20,
            )
        )

        assert not np.any(train_mask & val_mask)
        assert not np.any(train_mask & test_mask)
        assert not np.any(val_mask & test_mask)

        for trial_id in pd.unique(trial_ids):
            trial_mask = trial_ids == trial_id

            def selected_times(mask):
                return times[trial_mask & mask]

            train_times = selected_times(train_mask)
            val_times = selected_times(val_mask)
            test_times = selected_times(test_mask)

            train_val_separation = (
                float(val_times.min() - train_times.max())
                if len(train_times) and len(val_times)
                else np.nan
            )
            val_test_separation = (
                float(test_times.min() - val_times.max())
                if len(val_times) and len(test_times)
                else np.nan
            )

            split_rows.append({
                'suit_state': SUIT_STATE,
                'gravity': gravity,
                'muscle': muscle,
                'subject': int(subject),
                'trial_id': trial_id,
                'n_total_windows': int(trial_mask.sum()),
                'n_train': int((trial_mask & train_mask).sum()),
                'n_val': int((trial_mask & val_mask).sum()),
                'n_test': int((trial_mask & test_mask).sum()),
                'n_unused_after_purge': int(
                    trial_mask.sum()
                    - (trial_mask & train_mask).sum()
                    - (trial_mask & val_mask).sum()
                    - (trial_mask & test_mask).sum()
                ),
                'train_last_timestamp_s': (
                    float(train_times.max()) if len(train_times) else np.nan
                ),
                'val_first_timestamp_s': (
                    float(val_times.min()) if len(val_times) else np.nan
                ),
                'val_last_timestamp_s': (
                    float(val_times.max()) if len(val_times) else np.nan
                ),
                'test_first_timestamp_s': (
                    float(test_times.min()) if len(test_times) else np.nan
                ),
                'train_val_timestamp_separation_s': train_val_separation,
                'val_test_timestamp_separation_s': val_test_separation,
                'approx_train_val_signal_gap_s': (
                    train_val_separation - cfg.window_seconds
                    if np.isfinite(train_val_separation) else np.nan
                ),
                'approx_val_test_signal_gap_s': (
                    val_test_separation - cfg.window_seconds
                    if np.isfinite(val_test_separation) else np.nan
                ),
            })

split_audit = pd.DataFrame(split_rows)
split_audit.to_csv(
    os.path.join(RESULTS_ROOT, 'chronological_split_audit.csv'),
    index=False,
)

split_display = (
    split_audit
    .drop_duplicates([
        'gravity', 'subject', 'trial_id',
        'n_train', 'n_val', 'n_test',
    ])
    [[
        'gravity', 'subject', 'trial_id',
        'n_total_windows', 'n_train', 'n_val', 'n_test',
        'n_unused_after_purge',
        'train_val_timestamp_separation_s',
        'val_test_timestamp_separation_s',
    ]]
)
display(split_display)

## 8. Exact model comparison

All four models below receive the same participant-specific chronological
partitions. Ridge, Random Forest, and XGBoost hyperparameters are selected
using validation MAE. The selected model is refitted using training plus
validation data and evaluated once on the held-out test section.

The training-mean predictor has no hyperparameters and is fitted directly
on training plus validation targets. It is the minimum benchmark needed
to show whether a learned sensor-to-EMG mapping adds predictive value.

In [ ]:
def metric_record(y_true, prediction):
    metrics = compute_metrics(y_true, prediction)
    true_flat = np.asarray(y_true).reshape(-1)
    p05, p95 = np.percentile(true_flat, [5, 95])
    robust_range = float(p95 - p05)

    return {
        **metrics,
        'target_mean_test': float(np.mean(true_flat)),
        'target_std_test': float(np.std(true_flat)),
        'target_p05_test': float(p05),
        'target_p95_test': float(p95),
        'target_robust_range_test': robust_range,
        'nmae_p05_p95': (
            metrics['mae'] / robust_range
            if robust_range > 1e-12 else np.nan
        ),
        'nrmse_p05_p95': (
            metrics['rmse'] / robust_range
            if robust_range > 1e-12 else np.nan
        ),
    }


def save_model_result(
    rows,
    model_name,
    model,
    prediction,
    y_test,
    times_test,
    trial_ids_test,
    model_dir,
    muscle,
    gravity,
    subject,
    n_train,
    n_val,
    n_test,
    selected_parameters,
    validation_mae,
    feature_names,
):
    os.makedirs(model_dir, exist_ok=True)
    save_prediction_artifact(
        y_test,
        prediction,
        times_test,
        model_dir,
        f'{muscle} | {gravity} | {SUIT_STATE} | {model_name} | subject {subject}',
        trial_ids_test,
    )

    if model_name != 'training_mean':
        save_importance_artifact(
            model_name,
            model,
            feature_names,
            model_dir,
        )

    rows.append({
        'suit_state': SUIT_STATE,
        'gravity': gravity,
        'muscle': muscle,
        'subject': int(subject),
        'model': model_name,
        'n_train': int(n_train),
        'n_val': int(n_val),
        'n_test': int(n_test),
        'selection_metric': (
            'validation_mae'
            if model_name != 'training_mean'
            else 'not_applicable'
        ),
        'validation_mae': validation_mae,
        'selected_parameters': json.dumps(
            selected_parameters,
            sort_keys=True,
        ),
        **metric_record(y_test, prediction),
    })


def run_exact_comparison(data, gravity, muscle):
    rows = []

    for subject in np.unique(data.subjects):
        subject_mask = data.subjects == subject
        x = data.X_features[subject_mask]
        y = data.y[subject_mask]
        trial_ids = data.trial_ids[subject_mask]
        times = data.window_times[subject_mask]

        train_mask, val_mask, test_mask = (
            choose_subject_specific_masks(
                trial_ids,
                test_fraction=0.20,
                val_fraction=0.20,
                purge_windows=20,
            )
        )

        assert train_mask.sum() >= 2
        assert val_mask.sum() >= 1
        assert test_mask.sum() >= 1
        assert not np.any(train_mask & val_mask)
        assert not np.any(train_mask & test_mask)
        assert not np.any(val_mask & test_mask)

        x_train, y_train = x[train_mask], y[train_mask]
        x_val, y_val = x[val_mask], y[val_mask]
        x_test, y_test = x[test_mask], y[test_mask]

        (
            x_train_s,
            y_train_s,
            x_val_s,
            y_val_s,
            x_test_s,
            _,
            y_scaler,
        ) = scale_feature_matrix(
            x_train,
            y_train,
            x_val,
            y_val,
            x_test,
        )

        x_fit_s = np.vstack([x_train_s, x_val_s])
        y_fit_raw = np.vstack([y_train, y_val])
        y_fit_s = np.vstack([y_train_s, y_val_s])

        subject_root = os.path.join(
            RESULTS_ROOT,
            gravity,
            muscle,
            f'subject_{int(subject)}',
        )

        # 1. Training-mean baseline
        dummy = DummyRegressor(strategy='mean')
        dummy.fit(x_fit_s, y_fit_raw.reshape(-1))
        dummy_prediction = dummy.predict(x_test_s).reshape(-1, 1)
        save_model_result(
            rows=rows,
            model_name='training_mean',
            model=dummy,
            prediction=dummy_prediction,
            y_test=y_test,
            times_test=times[test_mask],
            trial_ids_test=trial_ids[test_mask],
            model_dir=os.path.join(subject_root, 'training_mean'),
            muscle=muscle,
            gravity=gravity,
            subject=subject,
            n_train=train_mask.sum(),
            n_val=val_mask.sum(),
            n_test=test_mask.sum(),
            selected_parameters={'strategy': 'mean'},
            validation_mae=np.nan,
            feature_names=data.feature_names,
        )

        # 2. Ridge baseline
        ridge_selected, ridge_meta = fit_best_ridge(
            x_train_s,
            y_train_s,
            x_val_s,
            y_val_s,
            cfg.ridge_alphas,
            cfg.seed,
        )
        ridge_val_scaled = ridge_selected.predict(x_val_s).reshape(-1, 1)
        ridge_val = y_scaler.inverse_transform(ridge_val_scaled)
        ridge_val_mae = float(mean_absolute_error(y_val, ridge_val))
        ridge_final = refit_ridge(
            ridge_meta['alpha'],
            x_fit_s,
            y_fit_s,
            cfg.seed,
        )
        ridge_prediction = y_scaler.inverse_transform(
            ridge_final.predict(x_test_s).reshape(-1, 1)
        )
        save_model_result(
            rows=rows,
            model_name='ridge',
            model=ridge_final,
            prediction=ridge_prediction,
            y_test=y_test,
            times_test=times[test_mask],
            trial_ids_test=trial_ids[test_mask],
            model_dir=os.path.join(subject_root, 'ridge'),
            muscle=muscle,
            gravity=gravity,
            subject=subject,
            n_train=train_mask.sum(),
            n_val=val_mask.sum(),
            n_test=test_mask.sum(),
            selected_parameters={'alpha': float(ridge_meta['alpha'])},
            validation_mae=ridge_val_mae,
            feature_names=data.feature_names,
        )

        # 3. Random Forest baseline
        rf_selected, rf_meta = fit_best_random_forest(
            x_train_s,
            y_train,
            x_val_s,
            y_val,
            cfg.rf_grid,
            cfg.seed,
        )
        rf_val = rf_selected.predict(x_val_s).reshape(-1, 1)
        rf_val_mae = float(mean_absolute_error(y_val, rf_val))
        rf_params = {
            key: rf_meta[key]
            for key in [
                'n_estimators',
                'max_depth',
                'min_samples_leaf',
            ]
        }
        rf_final = refit_random_forest(
            rf_params,
            x_fit_s,
            y_fit_raw,
            cfg.seed,
        )
        rf_prediction = rf_final.predict(x_test_s).reshape(-1, 1)
        save_model_result(
            rows=rows,
            model_name='random_forest',
            model=rf_final,
            prediction=rf_prediction,
            y_test=y_test,
            times_test=times[test_mask],
            trial_ids_test=trial_ids[test_mask],
            model_dir=os.path.join(subject_root, 'random_forest'),
            muscle=muscle,
            gravity=gravity,
            subject=subject,
            n_train=train_mask.sum(),
            n_val=val_mask.sum(),
            n_test=test_mask.sum(),
            selected_parameters=rf_params,
            validation_mae=rf_val_mae,
            feature_names=data.feature_names,
        )

        # 4. XGBoost
        xgb_selected, xgb_meta = fit_best_xgboost(
            x_train_s,
            y_train,
            x_val_s,
            y_val,
            cfg.xgb_grid,
            cfg.seed,
        )
        xgb_val = xgb_selected.predict(x_val_s).reshape(-1, 1)
        xgb_val_mae = float(mean_absolute_error(y_val, xgb_val))
        xgb_params = {
            key: xgb_meta[key]
            for key in [
                'n_estimators',
                'max_depth',
                'learning_rate',
                'subsample',
                'colsample_bytree',
            ]
        }
        xgb_final = refit_xgboost(
            xgb_params,
            x_fit_s,
            y_fit_raw,
            cfg.seed,
        )
        xgb_prediction = xgb_final.predict(x_test_s).reshape(-1, 1)
        save_model_result(
            rows=rows,
            model_name='xgboost',
            model=xgb_final,
            prediction=xgb_prediction,
            y_test=y_test,
            times_test=times[test_mask],
            trial_ids_test=trial_ids[test_mask],
            model_dir=os.path.join(subject_root, 'xgboost'),
            muscle=muscle,
            gravity=gravity,
            subject=subject,
            n_train=train_mask.sum(),
            n_val=val_mask.sum(),
            n_test=test_mask.sum(),
            selected_parameters=xgb_params,
            validation_mae=xgb_val_mae,
            feature_names=data.feature_names,
        )

    return pd.DataFrame(rows)

## 9. Run all seven muscles and both gravity levels

This is the computationally expensive cell. It fits the validation grids
separately for every participant, muscle, and gravity level.

In [ ]:
model_frames = []

for muscle in cfg.target_muscles:
    for gravity in ['Earth', 'Moon']:
        print(f'Running {SUIT_STATE}: {muscle} — {gravity}')
        model_frames.append(
            run_exact_comparison(
                data=datasets[(muscle, gravity)],
                gravity=gravity,
                muscle=muscle,
            )
        )

model_metrics = pd.concat(model_frames, ignore_index=True)
model_metrics.to_csv(
    os.path.join(
        RESULTS_ROOT,
        'model_comparison_subject_metrics.csv',
    ),
    index=False,
)

print('Completed model fits:', len(model_metrics))
display(model_metrics.head())

## 10. Summarise models and calculate paired XGBoost improvements

In [ ]:
model_summary = (
    model_metrics
    .groupby(
        ['suit_state', 'gravity', 'muscle', 'model'],
        as_index=False,
    )
    .agg(
        participants=('subject', 'nunique'),
        median_r2=('r2', 'median'),
        q1_r2=('r2', lambda x: x.quantile(0.25)),
        q3_r2=('r2', lambda x: x.quantile(0.75)),
        median_correlation=('corr', 'median'),
        median_nrmse=('nrmse_p05_p95', 'median'),
        median_nmae=('nmae_p05_p95', 'median'),
    )
)

model_summary.to_csv(
    os.path.join(RESULTS_ROOT, 'model_comparison_summary.csv'),
    index=False,
)

keys = ['suit_state', 'gravity', 'muscle', 'subject']
xgb_rows = model_metrics[
    model_metrics['model'].eq('xgboost')
][keys + ['r2', 'nrmse_p05_p95']].rename(columns={
    'r2': 'xgboost_r2',
    'nrmse_p05_p95': 'xgboost_nrmse',
})

paired_frames = []
for baseline in ['training_mean', 'ridge', 'random_forest']:
    baseline_rows = model_metrics[
        model_metrics['model'].eq(baseline)
    ][keys + ['r2', 'nrmse_p05_p95']].rename(columns={
        'r2': 'baseline_r2',
        'nrmse_p05_p95': 'baseline_nrmse',
    })

    paired = xgb_rows.merge(baseline_rows, on=keys, validate='one_to_one')
    paired['baseline'] = baseline
    paired['delta_r2_xgboost_minus_baseline'] = (
        paired['xgboost_r2'] - paired['baseline_r2']
    )
    paired['delta_nrmse_baseline_minus_xgboost'] = (
        paired['baseline_nrmse'] - paired['xgboost_nrmse']
    )
    paired['xgboost_better_r2'] = (
        paired['delta_r2_xgboost_minus_baseline'] > 0
    )
    paired['xgboost_better_nrmse'] = (
        paired['delta_nrmse_baseline_minus_xgboost'] > 0
    )
    paired_frames.append(paired)

paired_comparison = pd.concat(paired_frames, ignore_index=True)
paired_summary = (
    paired_comparison
    .groupby(
        ['suit_state', 'gravity', 'muscle', 'baseline'],
        as_index=False,
    )
    .agg(
        participants=('subject', 'nunique'),
        median_delta_r2=(
            'delta_r2_xgboost_minus_baseline', 'median'
        ),
        median_delta_nrmse=(
            'delta_nrmse_baseline_minus_xgboost', 'median'
        ),
        xgboost_r2_wins=('xgboost_better_r2', 'sum'),
        xgboost_nrmse_wins=('xgboost_better_nrmse', 'sum'),
    )
)

paired_comparison.to_csv(
    os.path.join(RESULTS_ROOT, 'paired_xgboost_comparisons.csv'),
    index=False,
)
paired_summary.to_csv(
    os.path.join(RESULTS_ROOT, 'paired_xgboost_summary.csv'),
    index=False,
)

xgboost_summary = model_summary[
    model_summary['model'].eq('xgboost')
].copy()
xgboost_summary.to_csv(
    os.path.join(RESULTS_ROOT, 'xgboost_paper_summary.csv'),
    index=False,
)

display(model_summary)
display(paired_summary)

## 11. Audit extreme metrics and low-variance test targets

Flags are diagnostic only. A flagged participant must not be deleted
automatically. First verify the original channel, units, electrode/gain,
file association, saturation, missing data, and preprocessing. Any
exclusion requires a documented data-quality reason followed by a full
rerun.

In [ ]:
xgb_diagnostics = model_metrics[
    model_metrics['model'].eq('xgboost')
].copy()

group_median_range = (
    xgb_diagnostics
    .groupby(['gravity', 'muscle'])[
        'target_robust_range_test'
    ]
    .transform('median')
)
xgb_diagnostics['range_relative_to_group_median'] = (
    xgb_diagnostics['target_robust_range_test']
    / group_median_range.replace(0, np.nan)
)
xgb_diagnostics['low_test_range_audit_flag'] = (
    xgb_diagnostics['range_relative_to_group_median'] < 0.10
)
xgb_diagnostics['extreme_negative_r2_audit_flag'] = (
    xgb_diagnostics['r2'] < -1.0
)
xgb_diagnostics['nrmse_above_one_audit_flag'] = (
    xgb_diagnostics['nrmse_p05_p95'] > 1.0
)

audit_columns = [
    'suit_state', 'gravity', 'muscle', 'subject',
    'r2', 'corr', 'rmse', 'mae',
    'target_mean_test', 'target_std_test',
    'target_p05_test', 'target_p95_test',
    'target_robust_range_test',
    'nrmse_p05_p95', 'nmae_p05_p95',
    'range_relative_to_group_median',
    'low_test_range_audit_flag',
    'extreme_negative_r2_audit_flag',
    'nrmse_above_one_audit_flag',
]
xgb_diagnostics = xgb_diagnostics[audit_columns]
xgb_diagnostics.to_csv(
    os.path.join(
        RESULTS_ROOT,
        'xgboost_subject_metrics_with_diagnostics.csv',
    ),
    index=False,
)

flagged_results = xgb_diagnostics[
    xgb_diagnostics[[
        'low_test_range_audit_flag',
        'extreme_negative_r2_audit_flag',
        'nrmse_above_one_audit_flag',
    ]].any(axis=1)
].copy()
flagged_results.to_csv(
    os.path.join(RESULTS_ROOT, 'flagged_results_for_audit.csv'),
    index=False,
)

# Source-level EMG amplitude audit, independent of model predictions.
source_amplitude_rows = []
for muscle in cfg.target_muscles:
    muscle_column = resolve_muscle_col(emg_df, muscle)
    for gravity in ['Earth', 'Moon']:
        for subject in sorted(emg_df['Subject'].unique()):
            raw_values = emg_df.loc[
                emg_df['Subject'].eq(subject)
                & emg_df['Gravity'].astype(str).str.lower().eq(
                    gravity.lower()
                ),
                muscle_column,
            ].to_numpy(dtype=float)

            nonfinite_count = int((~np.isfinite(raw_values)).sum())
            values = raw_values[np.isfinite(raw_values)]

            if not len(values):
                continue

            p01, p05, p50, p95, p99 = np.percentile(
                values, [1, 5, 50, 95, 99]
            )
            source_amplitude_rows.append({
                'suit_state': SUIT_STATE,
                'gravity': gravity,
                'muscle': muscle,
                'subject': int(subject),
                'samples': len(values),
                'mean': float(np.mean(values)),
                'std': float(np.std(values)),
                'p01': float(p01),
                'p05': float(p05),
                'median': float(p50),
                'p95': float(p95),
                'p99': float(p99),
                'robust_range_p05_p95': float(p95 - p05),
                'nonfinite_count': nonfinite_count,
            })

source_amplitude_audit = pd.DataFrame(source_amplitude_rows)
source_group_median = (
    source_amplitude_audit
    .groupby(['gravity', 'muscle'])[
        'robust_range_p05_p95'
    ]
    .transform('median')
)
source_amplitude_audit['range_ratio_to_group_median'] = (
    source_amplitude_audit['robust_range_p05_p95']
    / source_group_median.replace(0, np.nan)
)
source_amplitude_audit['scale_audit_flag'] = (
    (source_amplitude_audit['range_ratio_to_group_median'] < 0.10)
    | (source_amplitude_audit['range_ratio_to_group_median'] > 10.0)
)
source_amplitude_audit.to_csv(
    os.path.join(RESULTS_ROOT, 'source_emg_amplitude_audit.csv'),
    index=False,
)

print('Flagged model results:', len(flagged_results))
display(flagged_results)
display(
    source_amplitude_audit[
        source_amplitude_audit['scale_audit_flag']
    ]
)

## 12. Participant-level and model-comparison figures

In [ ]:
FIGURE_ROOT = os.path.join(RESULTS_ROOT, 'analysis_figures')
os.makedirs(FIGURE_ROOT, exist_ok=True)

muscle_order = [
    'Soleus',
    'Tibialis',
    'GastroMed',
    'VastusMed',
    'VastusLat',
    'RectusFemoris',
    'BicepsFemoris',
]

subject_ids = sorted(xgb_diagnostics['subject'].unique())
subject_offsets = np.linspace(-0.14, 0.14, len(subject_ids))
subject_colors = plt.cm.tab10(np.linspace(0, 1, len(subject_ids)))

fig, axes = plt.subplots(1, 2, figsize=(17, 6), sharex=True)

for ax, gravity in zip(axes, ['Earth', 'Moon']):
    subset = xgb_diagnostics[
        xgb_diagnostics['gravity'].eq(gravity)
    ]

    for position, muscle in enumerate(muscle_order):
        muscle_data = subset[subset['muscle'].eq(muscle)]

        for offset, color, subject in zip(
            subject_offsets,
            subject_colors,
            subject_ids,
        ):
            row = muscle_data[muscle_data['subject'].eq(subject)]
            if row.empty:
                continue
            value = float(row['r2'].iloc[0])
            ax.scatter(
                position + offset,
                value,
                color=color,
                s=38,
                alpha=0.85,
                label=(
                    f'Participant {subject}'
                    if gravity == 'Earth' and position == 0
                    else None
                ),
            )
            if value < -2:
                ax.annotate(
                    f'{value:.1f}',
                    (position + offset, value),
                    xytext=(4, 2),
                    textcoords='offset points',
                    fontsize=7,
                )

        median_value = muscle_data['r2'].median()
        ax.scatter(
            position,
            median_value,
            marker='_',
            s=300,
            linewidth=3,
            color='black',
        )

    ax.axhline(0, color='grey', linestyle='--', linewidth=1)
    ax.set_yscale('symlog', linthresh=1)
    ax.set_title(f'{SUIT_STATE} — {gravity}')
    ax.set_ylabel(r'Participant test $R^2$')
    ax.set_xticks(range(len(muscle_order)))
    ax.set_xticklabels(muscle_order, rotation=45, ha='right')
    ax.grid(alpha=0.2)

axes[0].legend(fontsize=8, ncol=2)
fig.tight_layout()
fig.savefig(
    os.path.join(
        FIGURE_ROOT,
        f'{SUIT_STATE.lower()}_participant_r2_distributions.png',
    ),
    dpi=300,
    bbox_inches='tight',
)
plt.show()

# Median model performance; participant-level CSV remains authoritative.
model_order = [
    'training_mean', 'ridge', 'random_forest', 'xgboost'
]
fig, axes = plt.subplots(1, 2, figsize=(17, 6), sharex=True)

for ax, gravity in zip(axes, ['Earth', 'Moon']):
    gravity_summary = model_summary[
        model_summary['gravity'].eq(gravity)
    ]
    for model_name in model_order:
        values = []
        for muscle in muscle_order:
            row = gravity_summary[
                gravity_summary['muscle'].eq(muscle)
                & gravity_summary['model'].eq(model_name)
            ]
            values.append(
                float(row['median_r2'].iloc[0])
                if not row.empty else np.nan
            )
        ax.plot(
            range(len(muscle_order)),
            values,
            marker='o',
            linewidth=1.5,
            label=model_name.replace('_', ' ').title(),
        )

    ax.axhline(0, color='grey', linestyle='--', linewidth=1)
    ax.set_yscale('symlog', linthresh=1)
    ax.set_title(f'{SUIT_STATE} — {gravity}')
    ax.set_ylabel(r'Median participant test $R^2$')
    ax.set_xticks(range(len(muscle_order)))
    ax.set_xticklabels(muscle_order, rotation=45, ha='right')
    ax.grid(alpha=0.2)

axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(
    os.path.join(
        FIGURE_ROOT,
        f'{SUIT_STATE.lower()}_exact_model_comparison.png',
    ),
    dpi=300,
    bbox_inches='tight',
)
plt.show()

## 13. Export untitled representative-participant XGBoost traces

Participant 2 was selected objectively as the participant closest to the
cell-specific six-participant medians across the 28 muscle--gravity--suit
combinations. The traces remain illustrative; participant distributions
and all numerical conclusions use all six participants. These plots omit
an internal title so that the caption can be supplied in Overleaf.

In [ ]:
# Participant 2 is the objectively selected representative participant.
PLOT_SUBJECT = 2
PLOT_SECONDS = 12
PAPER_FIGURE_ROOT = os.path.join(RESULTS_ROOT, 'paper_figures')
os.makedirs(PAPER_FIGURE_ROOT, exist_ok=True)

for muscle in cfg.target_muscles:
    for gravity in ['Earth', 'Moon']:
        prediction_path = os.path.join(
            RESULTS_ROOT,
            gravity,
            muscle,
            f'subject_{PLOT_SUBJECT}',
            'xgboost',
            'predictions.csv',
        )
        prediction_data = pd.read_csv(prediction_path)
        prediction_data = prediction_data.sort_values(
            ['trial_id', 'time_s']
        )

        first_trial = prediction_data['trial_id'].iloc[0]
        plot_data = prediction_data[
            prediction_data['trial_id'].eq(first_trial)
        ].copy()
        plot_data = plot_data[
            plot_data['time_s'] <= PLOT_SECONDS
        ]

        stem = (
            f'{muscle.lower()}_subject{PLOT_SUBJECT}_'
            f'{gravity.lower()}_{SUIT_STATE.lower()}_'
            'xgboost_true_vs_predicted'
        )
        plot_data.to_csv(
            os.path.join(PAPER_FIGURE_ROOT, stem + '.csv'),
            index=False,
        )

        fig, ax = plt.subplots(figsize=(12, 4.5))
        ax.plot(
            plot_data['time_s'],
            plot_data['true_target'],
            color='tab:blue',
            linewidth=1.8,
            label='Measured EMG-window target',
        )
        ax.plot(
            plot_data['time_s'],
            plot_data['pred_target'],
            color='tab:red',
            linestyle='--',
            linewidth=1.8,
            label='XGBoost prediction',
        )
        ax.set_xlabel('Time within held-out test section (s)')
        ax.set_ylabel('EMG-envelope window mean (a.u.)')
        ax.grid(alpha=0.2)
        ax.legend(frameon=False)
        fig.tight_layout()
        fig.savefig(
            os.path.join(PAPER_FIGURE_ROOT, stem + '.png'),
            dpi=300,
            bbox_inches='tight',
        )
        fig.savefig(
            os.path.join(PAPER_FIGURE_ROOT, stem + '.pdf'),
            bbox_inches='tight',
        )
        plt.close(fig)

print('Paper figures saved to:', PAPER_FIGURE_ROOT)

## 14. Optional combined BAMon/BAMoff participant distribution

Run this cell after both complete notebooks have finished. If the other
condition has not been run yet, the cell prints the missing path and stops
without affecting the present results.

In [ ]:
COMPLETE_ROOTS = {
    'BAMon': (
        '/content/drive/MyDrive/'
        'results_BAMon_complete_analysis'
    ),
    'BAMoff': (
        '/content/drive/MyDrive/'
        'results_BAMoff_complete_analysis'
    ),
}

diagnostic_paths = {
    state: os.path.join(
        root,
        'xgboost_subject_metrics_with_diagnostics.csv',
    )
    for state, root in COMPLETE_ROOTS.items()
}

missing = [
    path for path in diagnostic_paths.values()
    if not os.path.isfile(path)
]

if missing:
    print('Run both notebooks before creating the combined figure.')
    print('Missing:')
    for path in missing:
        print('-', path)
else:
    combined = pd.concat(
        [
            pd.read_csv(path)
            for path in diagnostic_paths.values()
        ],
        ignore_index=True,
    )
    combined.to_csv(
        '/content/drive/MyDrive/'
        'participant_level_distributions_complete.csv',
        index=False,
    )

    fig, axes = plt.subplots(
        2, 2,
        figsize=(17, 11),
        sharex=True,
    )

    panels = [
        ('BAMon', 'Earth'),
        ('BAMon', 'Moon'),
        ('BAMoff', 'Earth'),
        ('BAMoff', 'Moon'),
    ]

    for ax, (state, gravity) in zip(axes.flat, panels):
        subset = combined[
            combined['suit_state'].eq(state)
            & combined['gravity'].eq(gravity)
        ]

        for position, muscle in enumerate(muscle_order):
            muscle_data = subset[subset['muscle'].eq(muscle)]
            values = muscle_data.sort_values('subject')['r2'].to_numpy()
            offsets = np.linspace(-0.14, 0.14, len(values))
            ax.scatter(
                position + offsets,
                values,
                alpha=0.85,
                s=38,
            )
            ax.scatter(
                position,
                np.median(values),
                marker='_',
                s=300,
                linewidth=3,
                color='black',
            )
            for offset, value in zip(offsets, values):
                if value < -2:
                    ax.annotate(
                        f'{value:.1f}',
                        (position + offset, value),
                        xytext=(4, 2),
                        textcoords='offset points',
                        fontsize=7,
                    )

        ax.axhline(0, color='grey', linestyle='--', linewidth=1)
        ax.set_yscale('symlog', linthresh=1)
        ax.set_title(f'{state} — {gravity}')
        ax.set_ylabel(r'Participant test $R^2$')
        ax.set_xticks(range(len(muscle_order)))
        ax.set_xticklabels(muscle_order, rotation=45, ha='right')
        ax.grid(alpha=0.2)

    fig.tight_layout()
    combined_figure_path = (
        '/content/drive/MyDrive/'
        'participant_level_r2_distributions_complete.png'
    )
    fig.savefig(
        combined_figure_path,
        dpi=300,
        bbox_inches='tight',
    )
    plt.show()
    print('Combined figure:', combined_figure_path)

## 15. How to use these outputs in the manuscript

- `model_comparison_summary.csv` supplies the baseline table.
- `paired_xgboost_summary.csv` shows whether XGBoost improves on each
  baseline for the same participants and test sections.
- `software_versions.json`, `analysis_configuration.json`,
  `feature_names_124.csv`, and `chronological_split_audit.csv` support
  reproducibility.
- `xgboost_subject_metrics_with_diagnostics.csv` and the participant
  distribution figure resolve the missing participant-level reporting.
- `flagged_results_for_audit.csv` identifies recordings requiring source
  inspection. Flags are not exclusion decisions.

Do not state that XGBoost is the best model until the paired comparison
has been inspected. Do not describe the analysis as predicting a new
participant. The literature-novelty claim requires a separate structured
search; computational results cannot prove that no earlier study exists.

## 16. Final output list

In [ ]:
print('Completed. Output folder:')
print(RESULTS_ROOT)

for root, _, filenames in os.walk(RESULTS_ROOT):
    for filename in sorted(filenames):
        print(
            '-',
            os.path.relpath(
                os.path.join(root, filename),
                RESULTS_ROOT,
            ),
        )